# U2T02: SimCSE — Corrected Execution Notebook

This notebook runs the corrected repository end-to-end. It does not contain placeholder experiment results. All final numbers are generated by the actual runs.


## 0. Setup
Use a GPU runtime in Colab: **Runtime → Change runtime type → GPU**.


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Josch108/U2T02.git"
REPO_DIR = Path("/content/U2T02")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    os.chdir(REPO_DIR)
else:
    root = Path.cwd()
    if not (root / "src").exists() and (root.parent / "src").exists():
        root = root.parent
    os.chdir(root)

print("Working directory:", os.getcwd())


In [ ]:
# Keep Colab's preinstalled GPU-enabled PyTorch.\n# Install only the remaining pinned dependencies.\n!python -m pip install -q -r requirements.txt\n

In [ ]:
import subprocess, torch\n\nprint('Torch:', torch.__version__)\nprint('CUDA available:', torch.cuda.is_available())\n\nsubprocess.run(['nvidia-smi'], check=False)\n\nif not torch.cuda.is_available():\n    raise RuntimeError(\n        'CUDA is not available. Do not continue training. In Colab select an NVIDIA GPU runtime, reconnect/restart the runtime, and rerun from the top.'\n    )\n\nprint('GPU:', torch.cuda.get_device_name(0))\n

## 1. Provide the supplied SNLI file
The assignment file is intentionally not fabricated or downloaded from another source. Upload the provided `snli_train_100k.jsonl` if it is not already present.


In [ ]:
from pathlib import Path

data_path = Path("snli_train_100k.jsonl")
if not data_path.exists():
    if "google.colab" in sys.modules:
        from google.colab import files
        print("Upload snli_train_100k.jsonl")
        uploaded = files.upload()
    else:
        raise FileNotFoundError("Place snli_train_100k.jsonl in the repository root.")

assert data_path.exists(), "snli_train_100k.jsonl is still missing."
print("Dataset ready:", data_path)


## 2. Validate the supplied SNLI subset


In [ ]:
import importlib\nimport src.data_loader as data_loader\n\n# Important when the repository was updated inside the same Colab runtime.\nimportlib.invalidate_caches()\nimportlib.reload(data_loader)\n\nunsup_sentences, supervised_items = data_loader.load_snli_100k(\"snli_train_100k.jsonl\")\nhard_negatives = sum(item[\"contradiction\"] is not None for item in supervised_items)\n\nobserved = {\n    \"unique_sentences\": len(unsup_sentences),\n    \"supervised_pairs\": len(supervised_items),\n    \"hard_negatives\": hard_negatives,\n}\nexpected = {\n    \"unique_sentences\": 165_529,\n    \"supervised_pairs\": 33_351,\n    \"hard_negatives\": 9_488,\n}\n\nprint(\"Observed:\", observed)\nprint(\"Expected:\", expected)\n\nif observed != expected:\n    raise RuntimeError(f\"Dataset validation failed. Observed {observed}, expected {expected}\")\n\nprint(\"Dataset validation PASSED.\")\n

## 2.5 Preflight — verify every critical code path before long training
This fast smoke test checks CUDA, exact dataset counts, BERT loading, unsupervised forward pass, same-dropout ablation, supervised hard negatives, hard-negatives-OFF, and STS-B availability.


In [ ]:
!python -m src.preflight


## 3. Sanity-check the evaluator on STS-B dev
Test is not used here; checkpoint development remains based on dev only.


In [ ]:
!python -m src.verify_baselines --model both


## 4. Required run 1 — Unsupervised SimCSE


In [ ]:
!python -m src.train --mode unsupervised --data_path snli_train_100k.jsonl --batch_size 64 --lr 3e-5 --temperature 0.05 --epochs 1 --seed 42


## 5. Required run 2 — Same-dropout ablation


In [ ]:
!python -m src.train --mode unsupervised --data_path snli_train_100k.jsonl --same_dropout_ablation --batch_size 64 --lr 3e-5 --temperature 0.05 --epochs 1 --seed 42


## 6. Required run 3 — Supervised SimCSE with real hard negatives


In [ ]:
!python -m src.train --mode supervised --data_path snli_train_100k.jsonl --batch_size 64 --lr 5e-5 --temperature 0.05 --epochs 3 --seed 42


## 7. Required run 4 — Hard-negatives-OFF ablation


In [ ]:
!python -m src.train --mode supervised --data_path snli_train_100k.jsonl --no_hard_negatives --batch_size 64 --lr 5e-5 --temperature 0.05 --epochs 3 --seed 42


## 8. Generate benchmark, ablation deltas and plots


In [ ]:
!python -m src.make_report_artifacts --runs_dir runs --output_dir report/artifacts

print(Path("report/artifacts/benchmark_generated.md").read_text())
print(Path("report/artifacts/ablation_deltas_generated.md").read_text())


## 9. Final baseline test numbers
Run this only after the experiments are complete, when assembling the final benchmark.


In [ ]:
!python -m src.verify_baselines --model both --include_test


## 10. Publish and verify the selected model
Set your Hugging Face repository ID. Authenticate with `huggingface-cli login` or Colab secrets before running the command.


In [ ]:
HF_REPO_ID = "YOUR_USERNAME/simcse-bert-base-snli"
# Example after authentication:
# !python -m src.export_and_publish --run_dir runs/simcse_supervised_seed42 --export_dir st_best_model --push_to_hub --repo_id {HF_REPO_ID}


## 11. Final checklist
- Four `run_results.json` files exist.
- `runs/run_registry.jsonl` contains all runs.
- `report/artifacts/benchmark_generated.md` contains no MISSING rows.
- Retrieval examples were reviewed and at least one failure case was discussed.
- The selected Hub model was reloaded successfully and `hub_verification.json` reports `verified: true`.
